# NB07 — Multiclass ROC/AUC and Leakage-Safe Learning Curves

This notebook complements NB06 with two diagnostic analyses:

1. **Multiclass ROC/AUC (One-vs-Rest)** computed exclusively from previously saved out-of-fold (OOF) class probabilities. No model retraining is required for ROC analysis.
2. **Leakage-safe learning curves** for three representative models:
   - LDA → XGBoost (strong hybrid pipeline)
   - SVM-RBF (strong direct classical baseline)
   - FT-Transformer-style model (deep tabular comparator)

Learning curves reuse the exact outer stratified folds defined by seeds 2026, 2027 and 2028. For each outer fold, only the outer-training data are subsampled. The outer-test fold remains untouched. Classical-model hyperparameters are taken from the inner-CV choices already obtained in NB03/NB04 for the corresponding seed/fold; therefore, the learning-curve analysis is diagnostic and does not alter the primary performance estimates reported from NB03–NB06.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, ast, random, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

ROOT = Path('/content/drive/MyDrive/DRY_BEAN_HYBRID_Q1')
DATA = ROOT/'01_DATA'
RESULTS = ROOT/'03_RESULTS'
FIGURES = ROOT/'05_FIGURES'
TABLES = ROOT/'06_TABLES'
LOGS = ROOT/'07_LOGS'

OUT = RESULTS/'NB07_ROC_LEARNING'
FIG = FIGURES/'NB07_ROC_LEARNING'
TAB = TABLES/'NB07_ROC_LEARNING'
LOG = LOGS/'NB07_ROC_LEARNING'
for p in [OUT, FIG, TAB, LOG]:
    p.mkdir(parents=True, exist_ok=True)

DATASET = DATA/'INIAP_Dataset.xlsx'
OOF_FILE = RESULTS/'NB06_STATS'/'all_oof_predictions.csv'
BASE_PARAMS = RESULTS/'NB03_BASELINES'/'baseline_best_params.csv'
HYB_PARAMS = RESULTS/'NB04_HYBRIDS'/'hybrid_best_params.csv'

SEEDS = [2026, 2027, 2028]
OUTER_FOLDS = 5
TRAIN_FRACTIONS = [0.20, 0.40, 0.60, 0.80, 1.00]
TARGET = 'Class'
CLASSES = ['INIAP 420','INIAP 425','INIAP 481','INIAP 485']

for p in [DATASET, OOF_FILE, BASE_PARAMS, HYB_PARAMS]:
    assert p.exists(), f'Missing required file: {p}'

print('ROOT:', ROOT)
print('Dataset:', DATASET)
print('OOF:', OOF_FILE)


In [ ]:
!pip -q install xgboost


In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize, StandardScaler, LabelEncoder
from sklearn.metrics import roc_curve, auc, roc_auc_score, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from xgboost import XGBClassifier

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('NOTE: ROC analysis is fine on CPU; GPU is strongly recommended for FT-Transformer learning curves.')


## 1. Multiclass ROC/AUC from OOF probabilities

ROC curves are generated using a One-vs-Rest formulation. AUC values are computed separately for each class and as macro- and micro-averages for each seed. The main LDA-XGBoost ROC figure averages interpolated seed-specific ROC curves so repeated-seed predictions are not pooled as if they were independent observations.


In [ ]:
preds = pd.read_csv(OOF_FILE)
prob_cols = [f'prob_{c}' for c in CLASSES]
assert set(prob_cols).issubset(preds.columns), 'Missing OOF probability columns.'

models = sorted(preds['model'].unique())
print('Models:', models)

roc_rows = []
roc_curves = {}

grid = np.linspace(0,1,1001)

for model in models:
    roc_curves[model] = {}
    for seed in SEEDS:
        g = preds[(preds.model==model) & (preds.seed==seed)].drop_duplicates('row_id').sort_values('row_id')
        assert len(g)==4000, (model, seed, len(g))
        y_true = g['y_true'].astype(str).to_numpy()
        P = g[prob_cols].to_numpy(float)
        P = np.clip(P, 1e-12, 1.0)
        P = P / P.sum(axis=1, keepdims=True)
        Y = label_binarize(y_true, classes=CLASSES)

        macro_auc = roc_auc_score(Y, P, average='macro')
        micro_auc = roc_auc_score(Y, P, average='micro')
        weighted_auc = roc_auc_score(Y, P, average='weighted')

        row = {
            'model': model, 'seed': seed,
            'auc_macro_ovr': macro_auc,
            'auc_micro_ovr': micro_auc,
            'auc_weighted_ovr': weighted_auc
        }

        class_interp = []
        for k, cls in enumerate(CLASSES):
            fpr, tpr, _ = roc_curve(Y[:,k], P[:,k])
            cls_auc = auc(fpr,tpr)
            row[f'auc_{cls}'] = cls_auc
            interp = np.interp(grid, fpr, tpr)
            interp[0] = 0.0
            interp[-1] = 1.0
            class_interp.append(interp)
            roc_curves[model].setdefault(cls, []).append(interp)

        # macro ROC for this seed: mean of interpolated class ROCs
        macro_tpr = np.mean(class_interp, axis=0)
        roc_curves[model].setdefault('macro', []).append(macro_tpr)

        # micro ROC
        fpr_micro, tpr_micro, _ = roc_curve(Y.ravel(), P.ravel())
        interp_micro = np.interp(grid, fpr_micro, tpr_micro)
        interp_micro[0] = 0.0
        interp_micro[-1] = 1.0
        roc_curves[model].setdefault('micro', []).append(interp_micro)

        roc_rows.append(row)

roc_df = pd.DataFrame(roc_rows)
roc_df.to_csv(TAB/'roc_auc_by_model_seed.csv', index=False)

roc_summary = roc_df.groupby('model').agg(
    mean_macro_auc=('auc_macro_ovr','mean'),
    sd_macro_auc=('auc_macro_ovr','std'),
    mean_micro_auc=('auc_micro_ovr','mean'),
    sd_micro_auc=('auc_micro_ovr','std'),
    mean_weighted_auc=('auc_weighted_ovr','mean'),
    sd_weighted_auc=('auc_weighted_ovr','std'),
).sort_values('mean_macro_auc', ascending=False)

roc_summary.to_csv(TAB/'roc_auc_summary.csv')
display(roc_summary)


In [ ]:
# Main publication ROC figure — LDA_XGBoost
main_model = 'LDA_XGBoost'
plt.figure(figsize=(7.2,6.2))

for cls in CLASSES:
    arr = np.vstack(roc_curves[main_model][cls])
    mean_tpr = arr.mean(axis=0)
    auc_mean = roc_df.loc[roc_df.model==main_model, f'auc_{cls}'].mean()
    plt.plot(grid, mean_tpr, label=f'{cls} (AUC={auc_mean:.4f})')

macro_arr = np.vstack(roc_curves[main_model]['macro'])
micro_arr = np.vstack(roc_curves[main_model]['micro'])
plt.plot(grid, macro_arr.mean(axis=0),
         label=f"Macro-average (AUC={roc_df.loc[roc_df.model==main_model,'auc_macro_ovr'].mean():.4f})",
         linewidth=2.5)
plt.plot(grid, micro_arr.mean(axis=0),
         label=f"Micro-average (AUC={roc_df.loc[roc_df.model==main_model,'auc_micro_ovr'].mean():.4f})",
         linewidth=2.5)
plt.plot([0,1],[0,1],'--',linewidth=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multiclass OOF ROC — LDA-XGBoost (mean across seeds)')
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig(FIG/'roc_multiclass_LDA_XGBoost.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Macro-AUC comparison across all models
p = roc_summary.reset_index()
plt.figure(figsize=(10,5.8))
plt.errorbar(np.arange(len(p)), p.mean_macro_auc, yerr=p.sd_macro_auc, fmt='o', capsize=4)
plt.xticks(np.arange(len(p)), p.model, rotation=55, ha='right')
plt.ylabel('Macro ROC-AUC (OvR)')
plt.title('Macro ROC-AUC from out-of-fold probabilities')
plt.ylim(max(0.5, p.mean_macro_auc.min()-0.02), 1.005)
plt.tight_layout()
plt.savefig(FIG/'macro_auc_all_models.png', dpi=300, bbox_inches='tight')
plt.show()


## 2. Leakage-safe learning curves

For each seed and outer fold, the training portion is stratified-subsampled at 20%, 40%, 60%, 80% and 100%. The corresponding outer-test fold is never used for preprocessing, dimensionality reduction, early stopping, or parameter selection.

For SVM-RBF and LDA-XGBoost, the hyperparameters already selected by the inner CV of the matching seed/fold in NB03/NB04 are reused. For FT-Transformer, the same architecture and internal 15% validation procedure used in NB05 are reused. These curves are diagnostic and should not replace the primary nested-CV estimates.


In [ ]:
df = pd.read_excel(DATASET).rename(columns={'AspectRation':'AspectRatio'})
X = df.drop(columns=[TARGET]).astype(np.float32)
le = LabelEncoder()
y = pd.Series(le.fit_transform(df[TARGET].astype(str)), index=df.index)
class_map = {i:c for i,c in enumerate(le.classes_)}
print(class_map)

base_params = pd.read_csv(BASE_PARAMS)
hyb_params = pd.read_csv(HYB_PARAMS)

def parse_params(s):
    if isinstance(s, dict):
        return s
    try:
        return json.loads(s)
    except Exception:
        return ast.literal_eval(s)

def params_for(model, seed, fold):
    src = hyb_params if model=='LDA_XGBoost' else base_params
    row = src[(src.model==model)&(src.seed==seed)&(src.outer_fold==fold)]
    assert len(row)==1, (model,seed,fold,len(row))
    return parse_params(row.iloc[0].best_params)

def stratified_fraction_indices(y_arr, frac, seed):
    if frac >= 0.999999:
        return np.arange(len(y_arr))
    idx = np.arange(len(y_arr))
    keep, _ = train_test_split(
        idx, train_size=frac, stratify=y_arr, random_state=seed
    )
    return np.sort(keep)

def make_svm(params, seed):
    C = params.get('clf__C', params.get('C', 1))
    gamma = params.get('clf__gamma', params.get('gamma', 'scale'))
    return Pipeline([
        ('scale', StandardScaler()),
        ('clf', SVC(kernel='rbf', C=C, gamma=gamma, probability=True, random_state=seed))
    ])

def make_lda_xgb(params, seed):
    n_comp = int(params.get('dr__n_components', 3))
    kw = {
        'n_estimators': int(params.get('clf__n_estimators', 300)),
        'max_depth': int(params.get('clf__max_depth', 3)),
        'learning_rate': float(params.get('clf__learning_rate', 0.03)),
        'subsample': float(params.get('clf__subsample', 1.0)),
        'colsample_bytree': float(params.get('clf__colsample_bytree', 1.0)),
    }
    return Pipeline([
        ('scale', StandardScaler()),
        ('dr', LinearDiscriminantAnalysis(n_components=n_comp)),
        ('clf', XGBClassifier(
            objective='multi:softprob', eval_metric='mlogloss',
            tree_method='hist', n_jobs=1, random_state=seed, **kw
        ))
    ])


In [ ]:
# Compact FT-Transformer-style implementation, matching NB05
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

class FTTransformer(nn.Module):
    def __init__(self,n_features,n_classes,d_token=64,n_heads=4,n_layers=2,dropout=0.1):
        super().__init__()
        self.weight=nn.Parameter(torch.randn(n_features,d_token)*0.02)
        self.bias=nn.Parameter(torch.zeros(n_features,d_token))
        self.cls=nn.Parameter(torch.zeros(1,1,d_token))
        layer=nn.TransformerEncoderLayer(
            d_model=d_token,nhead=n_heads,dim_feedforward=d_token*4,
            dropout=dropout,batch_first=True,activation='gelu',norm_first=True
        )
        self.encoder=nn.TransformerEncoder(layer,num_layers=n_layers)
        self.head=nn.Sequential(nn.LayerNorm(d_token),nn.Linear(d_token,n_classes))
    def forward(self,x):
        tok=x.unsqueeze(-1)*self.weight.unsqueeze(0)+self.bias.unsqueeze(0)
        cls=self.cls.expand(x.size(0),-1,-1)
        z=self.encoder(torch.cat([cls,tok],dim=1))
        return self.head(z[:,0])

class ArrDS(Dataset):
    def __init__(self,X,y):
        self.X=torch.tensor(X,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self,i): return self.X[i],self.y[i]

def fit_ft(Xtr,ytr,Xv,yv,n_classes,seed,max_epochs=200,patience=20):
    seed_everything(seed)
    model=FTTransformer(Xtr.shape[1],n_classes).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
    loss_fn=nn.CrossEntropyLoss()
    gen=torch.Generator(); gen.manual_seed(seed)
    loader=DataLoader(ArrDS(Xtr,ytr),batch_size=128,shuffle=True,generator=gen)
    vaX=torch.tensor(Xv,dtype=torch.float32,device=DEVICE)
    vay=np.asarray(yv)
    best=-1; best_state=None; stale=0
    for epoch in range(max_epochs):
        model.train()
        for xb,yb in loader:
            xb=xb.to(DEVICE); yb=yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss=loss_fn(model(xb),yb)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            pred=model(vaX).argmax(1).cpu().numpy()
        score=f1_score(vay,pred,average='macro')
        if score>best+1e-5:
            best=score; stale=0
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            stale+=1
        if stale>=patience:
            break
    model.load_state_dict(best_state)
    model.to(DEVICE)
    return model, float(best), int(epoch+1)


In [ ]:
lc_rows = []
checkpoint = OUT/'learning_curve_CHECKPOINT.csv'

for seed in SEEDS:
    outer = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=seed)
    for fold,(tr,te) in enumerate(outer.split(X,y),start=1):
        Xtr_full = X.iloc[tr].values
        ytr_full = y.iloc[tr].values
        Xte = X.iloc[te].values
        yte = y.iloc[te].values

        for frac in TRAIN_FRACTIONS:
            sub_local = stratified_fraction_indices(
                ytr_full, frac, seed=seed*1000 + fold*10 + int(frac*100)
            )
            Xsub = Xtr_full[sub_local]
            ysub = ytr_full[sub_local]
            n_train = len(sub_local)

            print(f'\nseed={seed} fold={fold} frac={frac:.2f} n={n_train}', flush=True)

            # SVM-RBF
            p = params_for('SVM_RBF', seed, fold)
            mdl = make_svm(p, seed+fold)
            t0=time.time()
            mdl.fit(Xsub,ysub)
            tr_pred=mdl.predict(Xsub)
            te_pred=mdl.predict(Xte)
            lc_rows.append({
                'model':'SVM_RBF','seed':seed,'outer_fold':fold,
                'train_fraction':frac,'n_train':n_train,
                'train_macro_f1':f1_score(ysub,tr_pred,average='macro'),
                'test_macro_f1':f1_score(yte,te_pred,average='macro'),
                'fit_seconds':time.time()-t0
            })

            # LDA-XGBoost
            p = params_for('LDA_XGBoost', seed, fold)
            mdl = make_lda_xgb(p, seed+fold)
            t0=time.time()
            mdl.fit(Xsub,ysub)
            tr_pred=mdl.predict(Xsub)
            te_pred=mdl.predict(Xte)
            lc_rows.append({
                'model':'LDA_XGBoost','seed':seed,'outer_fold':fold,
                'train_fraction':frac,'n_train':n_train,
                'train_macro_f1':f1_score(ysub,tr_pred,average='macro'),
                'test_macro_f1':f1_score(yte,te_pred,average='macro'),
                'fit_seconds':time.time()-t0
            })

            # FT-Transformer-style
            fold_seed = seed + fold + int(frac*1000)
            idx=np.arange(len(ysub))
            tri,vi=train_test_split(
                idx,test_size=.15,stratify=ysub,random_state=fold_seed
            )
            scaler=StandardScaler().fit(Xsub[tri])
            Xa=scaler.transform(Xsub[tri]).astype(np.float32)
            Xv=scaler.transform(Xsub[vi]).astype(np.float32)
            Xtest=scaler.transform(Xte).astype(np.float32)
            Xall=scaler.transform(Xsub).astype(np.float32)
            ya=ysub[tri]; yv=ysub[vi]

            t0=time.time()
            ft,best_val,n_ep=fit_ft(Xa,ya,Xv,yv,len(CLASSES),fold_seed)
            ft.eval()
            with torch.no_grad():
                tr_pred=ft(torch.tensor(Xall,dtype=torch.float32,device=DEVICE)).argmax(1).cpu().numpy()
                te_pred=ft(torch.tensor(Xtest,dtype=torch.float32,device=DEVICE)).argmax(1).cpu().numpy()
            lc_rows.append({
                'model':'FTTransformer','seed':seed,'outer_fold':fold,
                'train_fraction':frac,'n_train':n_train,
                'train_macro_f1':f1_score(ysub,tr_pred,average='macro'),
                'test_macro_f1':f1_score(yte,te_pred,average='macro'),
                'fit_seconds':time.time()-t0,
                'epochs':n_ep,'best_val_macro_f1':best_val
            })

            pd.DataFrame(lc_rows).to_csv(checkpoint,index=False)

lc = pd.DataFrame(lc_rows)
lc.to_csv(OUT/'learning_curve_by_fold.csv',index=False)
print('Learning-curve rows:',len(lc),'expected:',3*3*5*5)


In [ ]:
lc_summary = lc.groupby(['model','train_fraction']).agg(
    mean_n_train=('n_train','mean'),
    mean_train_macro_f1=('train_macro_f1','mean'),
    sd_train_macro_f1=('train_macro_f1','std'),
    mean_test_macro_f1=('test_macro_f1','mean'),
    sd_test_macro_f1=('test_macro_f1','std'),
    mean_fit_seconds=('fit_seconds','mean')
).reset_index()

lc_summary.to_csv(TAB/'learning_curve_summary.csv',index=False)
display(lc_summary)

for model in ['LDA_XGBoost','SVM_RBF','FTTransformer']:
    g=lc_summary[lc_summary.model==model].sort_values('mean_n_train')
    plt.figure(figsize=(7.2,5.5))
    plt.errorbar(g.mean_n_train,g.mean_train_macro_f1,yerr=g.sd_train_macro_f1,
                 marker='o',capsize=4,label='Training Macro-F1')
    plt.errorbar(g.mean_n_train,g.mean_test_macro_f1,yerr=g.sd_test_macro_f1,
                 marker='o',capsize=4,label='Outer-test Macro-F1')
    plt.xlabel('Training samples')
    plt.ylabel('Macro-F1')
    plt.title(f'Learning curve — {model}')
    plt.legend()
    plt.ylim(max(0.80, min(g.mean_test_macro_f1.min(),g.mean_train_macro_f1.min())-0.03),1.005)
    plt.tight_layout()
    plt.savefig(FIG/f'learning_curve_{model}.png',dpi=300,bbox_inches='tight')
    plt.show()

# Combined outer-test learning curve
plt.figure(figsize=(7.6,5.6))
for model in ['LDA_XGBoost','SVM_RBF','FTTransformer']:
    g=lc_summary[lc_summary.model==model].sort_values('mean_n_train')
    plt.errorbar(g.mean_n_train,g.mean_test_macro_f1,yerr=g.sd_test_macro_f1,
                 marker='o',capsize=4,label=model)
plt.xlabel('Training samples')
plt.ylabel('Outer-test Macro-F1')
plt.title('Learning-curve comparison')
plt.legend()
plt.tight_layout()
plt.savefig(FIG/'learning_curve_comparison_test.png',dpi=300,bbox_inches='tight')
plt.show()


In [ ]:
run_info = {
    'seeds': SEEDS,
    'outer_folds': OUTER_FOLDS,
    'train_fractions': TRAIN_FRACTIONS,
    'roc_source': str(OOF_FILE),
    'roc_formulation': 'One-vs-Rest multiclass ROC from OOF probabilities; per-seed AUCs; mean interpolated curves across seeds',
    'learning_curve_models': ['LDA_XGBoost','SVM_RBF','FTTransformer'],
    'learning_curve_note': (
        'Outer-test folds untouched. Classical hyperparameters reused from the matching '
        'seed/fold inner-CV selections of NB03/NB04. FT-Transformer uses the NB05 architecture '
        'with 15% validation inside each subsampled outer-training set. Learning curves are diagnostic.'
    ),
    'device': DEVICE,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
}
with open(LOG/'NB07_run_info.json','w') as f:
    json.dump(run_info,f,indent=2)

print('\nNB07 completed successfully.')
